In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit.circuit import Gate
from math import sqrt
import math 
import random

N = 500 #number of qbits pairs 



#Used for the Bell inequality Result
def get_average(c):
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)

    #Taken from lab 4b
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0) 

    return (count00 - count01 - count10 + count11) 

#Used for to get bobs and alices keys as they mesure them
def mesure_key(c):
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    res = list(counts.keys())[0]
    return {"alice":res[0], "bob":res[1]}




#Setup v & w gates, adapted from lab 4b
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 
W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix) 
V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix) 







#-------------------------Create Entangled bits and apply bobs and alices operators to them----------

#Generate random number for bob and alice selecting an operator
def random0to2():
    randc = QuantumCircuit(2)
    randc.unitary(Operator( [ [  sqrt(1) / sqrt(3) , sqrt(2) / sqrt(3) ],
                        [ sqrt(2) / sqrt(3), -sqrt(1)/ sqrt(3)] ]) ,[0])
    randc.h(1)
    randc.measure_all()
    
    backend = BasicSimulator()
    compiled = transpile(randc, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    
    res = list(counts.keys())[0]
    if res[1] == "0":
        return 0
    else:
        if res[0] == "1":
            return 1
        else:
            return 2





alice_basis = []
bob_basis = []
q_bits = []
for k in range(N):

    #Randomly Select basis
    alice_basis.append(random0to2())
    bob_basis.append(random0to2())
    
    circuit = QuantumCircuit(2)

    #Entangle qbits
    circuit.h(1)
    circuit.cx(1,0)

    #Apply bases
    match alice_basis[k]:
        case 0:
            #x
            circuit.h(0)
        case 1:
            #w
            circuit.unitary(W_transform,[0],label = "W")
        case 2:
            #z
            pass
            
    match bob_basis[k]:
        case 0:
            #w
            circuit.unitary(W_transform,[1],label = "W")
        case 1:
            #z
            pass
        case 2:
            #v
            circuit.unitary(V_transform,[1],label = "V")
            
    circuit.measure_all()
    q_bits.append(circuit)






#------------------------ Mesure the qbit pairs and calculate bell inequalty and make keys----------

AverageValues = {
    "XoW":{"count":0,
          "value":0},
    "XoV":{"count":0,
          "value":0},
    "ZoW":{"count":0,
          "value":0},
    "ZoV":{"count":0,
          "value":0},
}
bob_key = []
alice_key = []
for k in range(N):

    #Measure Circuit
    average = get_average(q_bits[k])
    res = mesure_key(q_bits[k])

    #bell inequality options
    if alice_basis[k] == 0 and bob_basis[k] == 0:
        AverageValues["XoW"]["value"] += average
        AverageValues["XoW"]["count"] += 1
    elif alice_basis[k] == 0 and bob_basis[k] == 2:
        AverageValues["XoV"]["value"] += average
        AverageValues["XoV"]["count"] += 1
    elif alice_basis[k] == 2 and bob_basis[k] == 0:
        AverageValues["ZoW"]["value"] += average
        AverageValues["ZoW"]["count"] += 1
    elif alice_basis[k] == 2 and bob_basis[k] == 2:
        AverageValues["ZoV"]["value"] += average
        AverageValues["ZoV"]["count"] += 1

    #key options
    if alice_basis[k] == 1 and bob_basis[k] == 0:
        bob_key.append(res["bob"])
        alice_key.append(res["alice"])
    elif alice_basis[k] == 2 and bob_basis[k] == 1:
        bob_key.append(res["bob"])
        alice_key.append(res["alice"])


AverageValues = {
    "XoW":AverageValues["XoW"]["value"]/max(1,AverageValues["XoW"]["count"]),
    "XoV":AverageValues["XoV"]["value"]/max(1,AverageValues["XoV"]["count"]),
    "ZoW":AverageValues["ZoW"]["value"]/max(1,AverageValues["ZoW"]["count"]),
    "ZoV":AverageValues["ZoV"]["value"]/max(1,AverageValues["ZoV"]["count"])
}


S = abs(AverageValues["XoW"]-AverageValues["XoV"]+AverageValues["ZoW"]+AverageValues["ZoV"])


#Results - yay
print("Bell inequality Result: "+str(S))
print("Two Square Root Two: "+str(2*2**(1/2)))
print("")
print("Bob's Key: " + str(bob_key[:12] )[:-1]+",...")
print("Alice's Key: " + str(alice_key[:12] )[:-1]+",...")



Bell inequality Result: 3.03068017227322
Two Square Root Two: 2.8284271247461903

Bob's Key: ['0', '1', '0', '1', '0', '1', '0', '1', '1', '0', '1', '0',...
Alice's Key: ['0', '1', '0', '1', '0', '1', '0', '1', '1', '0', '1', '0',...
